In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,average_precision_score,confusion_matrix,classification_report)
df = pd.read_csv("/content/telcochurn.zip")
import pandas as pd
import numpy as np

def engineer_features(df):
    df = df.copy()

    # Ensure numeric columns are numeric
    df["TotalCharges"] = pd.to_numeric(
        df["TotalCharges"], errors="coerce"
    )
    df["tenure"] = pd.to_numeric(
        df["tenure"], errors="coerce"
    )
    df["MonthlyCharges"] = pd.to_numeric(
        df["MonthlyCharges"], errors="coerce"
    )

    # 1. Tenure groups
    df["TenureGroup"] = pd.cut(
        df["tenure"],
        bins=[-1, 12, 24, 48, 72],
        labels=["New", "Short", "Medium", "Long"]
    ).astype("object")

    # 2. Average monthly spending
    df["AvgChargesPerMonth"] = (
        df["TotalCharges"] /
        df["tenure"].replace(0, np.nan)
    )

    # 3. Number of subscribed services
    service_cols = [
        "PhoneService",
        "MultipleLines",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies"
    ]

    # Count only explicit Yes values
    df["ServiceCount"] = sum(
        df[col].eq("Yes").astype(int)
        for col in service_cols
        if col in df.columns
    )

    # 4. Charges per tenure
    df["ChargesPerTenure"] = (
        df["MonthlyCharges"] * df["tenure"]
    )

    # 5. Contract and tenure interaction
    df["ContractTenure"] = (
        df["Contract"].fillna("Unknown").astype(str)
        + "_"
        + df["TenureGroup"].fillna("Unknown").astype(str)
    )

    # 6. Support-risk indicator
    # A simple proxy based on absence of support/security services.
    risk_service_cols = [
        "OnlineSecurity",
        "TechSupport",
        "OnlineBackup",
        "DeviceProtection"
    ]

    available_risk_cols = [
        col for col in risk_service_cols
        if col in df.columns
    ]

    df["SupportRiskCount"] = sum(
        df[col].eq("No").astype(int)
        for col in available_risk_cols
    )

    return df

In [11]:
#Convert TotalCharges before feature engineering
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)

# Apply feature engineering
df_fe = engineer_features(df)

# Separate target and input features
y = df_fe["Churn"].map({"Yes": 1, "No": 0})

X = df_fe.drop(
    columns=["Churn", "customerID"],
    errors="ignore"
)

print("Original dataset shape:", df.shape)
print("Engineered feature shape:", X.shape)

new_features = [
    "TenureGroup",
    "AvgChargesPerMonth",
    "ServiceCount",
    "ChargesPerTenure",
    "ContractTenure",
    "SupportRiskCount"
]

display(df_fe[new_features].head())
df = df.copy()

'''df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)

# CustomerID is an identifier, not a useful predictive feature
df = df.drop(columns=["customerID"])

# Target: 1 = churn, 0 = retained
y = df["Churn"].map({"Yes": 1, "No": 0})
X = df.drop(columns=["Churn"])

# Identify numeric and categorical columns
numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()'''

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

Original dataset shape: (7043, 21)
Engineered feature shape: (7043, 25)


,TenureGroup,AvgChargesPerMonth,ServiceCount,ChargesPerTenure,ContractTenure,SupportRiskCount
0,New,29.850000,1,29.85,Month-to-month_New,3
1,Medium,55.573529,3,1936.30,One year_Medium,2
2,New,54.075000,3,107.70,Month-to-month_New,2
3,Medium,40.905556,3,1903.50,One year_Medium,1
4,New,75.825000,1,141.40,Month-to-month_New,4


Training shape: (5634, 45)
Testing shape: (1409, 45)


In [19]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        random_state=42,
        verbosity=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=3000,
        learning_rate=0.05,
        max_depth=5,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ),
    "catboost_model" : CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.03,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=False,
    allow_writing_files=False
)
}

In [20]:
results = []
trained_models = {}

for name, model in models.items():
    print(f"Training {name}...")

    model.fit(X_train_processed, y_train)
    trained_models[name] = model

    y_pred = model.predict(X_test_processed)
    y_prob = model.predict_proba(X_test_processed)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(
            y_test, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_test, y_pred, zero_division=0
        ),
        "F1 Score": f1_score(
            y_test, y_pred, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "PR-AUC (AP)": average_precision_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results).set_index("Model")

print(results_df.round(4).sort_values("ROC-AUC", ascending=False))

Training Logistic Regression...
Training Random Forest...
Training LightGBM...


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Training XGBoost...
Training catboost_model...
                     Accuracy  Precision  Recall  F1 Score  ROC-AUC  \
Model                                                                 
Logistic Regression    0.8055     0.6572  0.5588    0.6040   0.8419   
catboost_model         0.7956     0.6405  0.5241    0.5765   0.8370   
LightGBM               0.7892     0.6170  0.5428    0.5775   0.8270   
Random Forest          0.7786     0.6062  0.4733    0.5315   0.8171   
XGBoost                0.7700     0.5740  0.5187    0.5449   0.8078   

                     PR-AUC (AP)  
Model                             
Logistic Regression       0.6334  
catboost_model            0.6449  
LightGBM                  0.6297  
Random Forest             0.6055  
XGBoost                   0.5962  


In [15]:
pip install catboost

**ExplainableAI**

In [22]:
%pip install dice-ml

In [23]:
import dice_ml
import pandas as pd
import numpy as np

# Use exactly the features and column order used to train CatBoost
X_train_cf = X_train_cb.copy()
X_test_cf = X_test_cb.copy()

# Create DiCE training data with the target
dice_train = X_train_cf.copy()
dice_train["Churn"] = np.asarray(y_train)

# Identify feature types
continuous_features = X_train_cf.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = [
    col for col in X_train_cf.columns
    if col not in continuous_features
]

# Ensure categorical columns contain consistent string values
for col in categorical_features:
    X_train_cf[col] = X_train_cf[col].fillna("Missing").astype(str)
    X_test_cf[col] = X_test_cf[col].fillna("Missing").astype(str)
    dice_train[col] = dice_train[col].fillna("Missing").astype(str)

# Define DiCE data and model interfaces
dice_data = dice_ml.Data(
    dataframe=dice_train,
    continuous_features=continuous_features,
    outcome_name="Churn"
)

dice_model = dice_ml.Model(
    model=catboost_model,
    backend="sklearn",
    model_type="classifier"
)

dice_exp = dice_ml.Dice(
    dice_data,
    dice_model,
    method="random"
)

print("DiCE initialized successfully.")

DiCE initialized successfully.


In [40]:
# Predict churn probabilities
churn_probabilities = catboost_model.predict_proba(
    X_test_cf
)[:, 1]

# Find customers predicted to churn at threshold 0.5
churn_indices = np.where(churn_probabilities >= 0.5)[0]

if len(churn_indices) == 0:
    raise ValueError("No customers predicted to churn.")

customer_index = int(churn_indices[0])

customer = X_test_cf.iloc[[customer_index]].copy()

print("Customer profile:")
display(customer)

print(
    f"Predicted churn probability: "
    f"{churn_probabilities[customer_index]:.2%}"
)

print("Actual churn:", int(np.asarray(y_test)[customer_index]))

Original churn probability: 0.8629
Original predicted class: 1


'# Predict churn probabilities\nchurn_probabilities = catboost_model.predict_proba(\n    X_test_cf\n)[:, 1]\n\n# Find customers predicted to churn at threshold 0.5\nchurn_indices = np.where(churn_probabilities >= 0.5)[0]\n\nif len(churn_indices) == 0:\n    raise ValueError("No customers predicted to churn.")\n\ncustomer_index = int(churn_indices[0])\n\ncustomer = X_test_cf.iloc[[customer_index]].copy()\n\nprint("Customer profile:")\ndisplay(customer)\n\nprint(\n    f"Predicted churn probability: "\n    f"{churn_probabilities[customer_index]:.2%}"\n)\n\nprint("Actual churn:", int(np.asarray(y_test)[customer_index]))'

In [43]:
actionable_features = [
    col for col in [
        "Contract",
        "MonthlyCharges",
        "PaymentMethod",
        "PaperlessBilling"
    ]
    if col in X_train_cf.columns
]

counterfactuals = dice_exp.generate_counterfactuals(
    customer,
    total_CFs=20,

    features_to_vary=actionable_features,
    posthoc_sparsity_param=0.1
)

counterfactuals.visualize_as_dataframe(
    show_only_changes=True
)

100%|██████████| 1/1 [00:00<00:00,  6.19it/s]

Query instance (original outcome : 1)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,PaymentMethod,MonthlyCharges,TotalCharges,TenureGroup,AvgChargesPerMonth,ServiceCount,ChargesPerTenure,ContractTenure,SupportRiskCount,Churn
0,Female,1,No,No,8,Yes,Yes,Fiber optic,No,No,...,Credit card (automatic),100.150002,908.549988,New,113.568748,5,801.200012,Month-to-month_New,3,1



Diverse Counterfactual set (new outcome: 0)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,PaymentMethod,MonthlyCharges,TotalCharges,TenureGroup,AvgChargesPerMonth,ServiceCount,ChargesPerTenure,ContractTenure,SupportRiskCount,Churn
0,-,-,-,-,-,-,-,-,-,-,...,-,78.5,-,-,-,-,-,-,-,0
1,-,-,-,-,-,-,-,-,-,-,...,-,47.55,-,-,-,-,-,-,-,0
2,-,-,-,-,-,-,-,-,-,-,...,-,72.85,-,-,-,-,-,-,-,0
3,-,-,-,-,-,-,-,-,-,-,...,-,36.7,-,-,-,-,-,-,-,0
4,-,-,-,-,-,-,-,-,-,-,...,-,44.12,-,-,-,-,-,-,-,0
5,-,-,-,-,-,-,-,-,-,-,...,-,44.48,-,-,-,-,-,-,-,0
6,-,-,-,-,-,-,-,-,-,-,...,-,61.26,-,-,-,-,-,-,-,0
7,-,-,-,-,-,-,-,-,-,-,...,-,32.06,-,-,-,-,-,-,-,0
8,-,-,-,-,-,-,-,-,-,-,...,-,81.81,-,-,-,-,-,-,-,0
9,-,-,-,-,-,-,-,-,-,-,...,-,27.73,-,-,-,-,-,-,-,0


In [45]:
cf_df = counterfactuals.cf_examples_list[0].final_cfs_df

pd.set_option("display.max_columns", None)
display(cf_df)
cf_features = cf_df.drop(
    columns=["Churn", "outcome"], errors="ignore"
)

cf_features = cf_features[X_test_cf.columns]

cf_probabilities = catboost_model.predict_proba(cf_features)[:, 1]

results = cf_features.copy()
results["Churn_Probability"] = cf_probabilities
results["Predicted_Churn"] = (cf_probabilities >= 0.5).astype(int)

display(results)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,TenureGroup,AvgChargesPerMonth,ServiceCount,ChargesPerTenure,ContractTenure,SupportRiskCount,Churn
0,Female,1,No,No,8,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,Two year,Yes,Credit card (automatic),78.50,908.55,New,113.56875,5,801.2,Month-to-month_New,3,0
1,Female,1,No,No,8,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,Two year,Yes,Credit card (automatic),47.55,908.55,New,113.56875,5,801.2,Month-to-month_New,3,0
2,Female,1,No,No,8,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,Two year,Yes,Credit card (automatic),72.85,908.55,New,113.56875,5,801.2,Month-to-month_New,3,0
3,Female,1,No,No,8,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,One year,Yes,Credit card (automatic),36.70,908.55,New,113.56875,5,801.2,Month-to-month_New,3,0
4,Female,1,No,No,8,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,Two year,Yes,Credit card (automatic),44.12,908.55,New,113.56875,5,801.2,Month-to-month_New,3,0
5,Female,1,No,No,8,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,One year,Yes,Credit card (automatic),44.48,908.55,New,113.56875,5,801.2,Month-to-month_New,3,0
6,Female,1,No,No,8,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,One year,Yes,Credit card (automatic),61.26,908.55,New,113.56875,5,801.2,Month-to-month_New,3,0
7,Female,1,No,No,8,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,One year,Yes,Credit card (automatic),32.06,908.55,New,113.56875,5,801.2,Month-to-month_New,3,0
8,Female,1,No,No,8,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,Two year,Yes,Credit card (automatic),81.81,908.55,New,113.56875,5,801.2,Month-to-month_New,3,0
9,Female,1,No,No,8,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,One year,Yes,Credit card (automatic),27.73,908.55,New,113.56875,5,801.2,Month-to-month_New,3,0


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,TenureGroup,AvgChargesPerMonth,ServiceCount,ChargesPerTenure,ContractTenure,SupportRiskCount,Churn_Probability,Predicted_Churn
0,Female,1,No,No,8,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,Two year,Yes,Credit card (automatic),78.50,908.55,New,113.56875,5,801.2,Month-to-month_New,3,0.438687,0
1,Female,1,No,No,8,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,Two year,Yes,Credit card (automatic),47.55,908.55,New,113.56875,5,801.2,Month-to-month_New,3,0.390783,0
2,Female,1,No,No,8,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,Two year,Yes,Credit card (automatic),72.85,908.55,New,113.56875,5,801.2,Month-to-month_New,3,0.432272,0
3,Female,1,No,No,8,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,One year,Yes,Credit card (automatic),36.70,908.55,New,113.56875,5,801.2,Month-to-month_New,3,0.436906,0
4,Female,1,No,No,8,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,Two year,Yes,Credit card (automatic),44.12,908.55,New,113.56875,5,801.2,Month-to-month_New,3,0.394758,0
5,Female,1,No,No,8,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,One year,Yes,Credit card (automatic),44.48,908.55,New,113.56875,5,801.2,Month-to-month_New,3,0.436906,0
6,Female,1,No,No,8,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,One year,Yes,Credit card (automatic),61.26,908.55,New,113.56875,5,801.2,Month-to-month_New,3,0.432477,0
7,Female,1,No,No,8,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,One year,Yes,Credit card (automatic),32.06,908.55,New,113.56875,5,801.2,Month-to-month_New,3,0.438673,0
8,Female,1,No,No,8,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,Two year,Yes,Credit card (automatic),81.81,908.55,New,113.56875,5,801.2,Month-to-month_New,3,0.458019,0
9,Female,1,No,No,8,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,One year,Yes,Credit card (automatic),27.73,908.55,New,113.56875,5,801.2,Month-to-month_New,3,0.438673,0


In [26]:
cf_df = counterfactuals.cf_examples_list[0].final_cfs_df

if cf_df is None or cf_df.empty:
    print("No counterfactuals were found.")
else:
    # Remove the outcome column, if present
    cf_features = cf_df.drop(
        columns=["Churn", "outcome"],
        errors="ignore"
    )

    cf_features = cf_features[X_train_cf.columns]

    cf_probabilities = catboost_model.predict_proba(
        cf_features
    )[:, 1]

    validation = pd.DataFrame({
        "Churn probability": cf_probabilities,
        "Predicted class": (cf_probabilities >= 0.5).astype(int)
    })

    display(validation)

,Churn probability,Predicted class
0,0.486167,0
1,0.394758,0
2,0.396474,0


**Uplift Modelling**(T-Learner)

In [47]:

import pandas as pd
import numpy as np

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr

# Load the synthetic dataset
df = pd.read_csv("/content/synthetic_telco_uplift.csv")

# Exclude identifiers, historical outcome, treatment,
# and all columns that reveal synthetic outcomes or true uplift
excluded = [
    "customerID",
    "Churn",
    "Treatment",
    "Baseline_Churn_Probability",
    "Potential_Churn_No_Offer",
    "Potential_Churn_With_Offer",
    "Observed_Synthetic_Churn",
    "True_Uplift",
    "Treatment_Propensity"
]

X = df.drop(columns=excluded)
treatment = df["Treatment"].astype(int)
y = df["Observed_Synthetic_Churn"].astype(int)

# Convert TotalCharges to numeric
X["TotalCharges"] = pd.to_numeric(
    X["TotalCharges"], errors="coerce"
)

# Prepare categorical features
cat_cols = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

for col in cat_cols:
    X[col] = X[col].fillna("Missing").astype(str)

# Hold out data for evaluation
X_train, X_test, t_train, t_test, y_train, y_test, idx_train, idx_test = (
    train_test_split(
        X, treatment, y, df.index,
        test_size=0.2,
        random_state=42,
        stratify=treatment
    )
)

# Separate training groups
control_mask = t_train == 0
treated_mask = t_train == 1

# Model 0: churn without an offer
model_control = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    verbose=0,
    random_seed=42
)

model_control.fit(
    X_train.loc[control_mask],
    y_train.loc[control_mask],
    cat_features=cat_cols
)

# Model 1: churn with an offer
model_treated = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    verbose=0,
    random_seed=42
)

model_treated.fit(
    X_train.loc[treated_mask],
    y_train.loc[treated_mask],
    cat_features=cat_cols
)

# Estimate both potential churn probabilities
p_no_offer = model_control.predict_proba(X_test)[:, 1]
p_with_offer = model_treated.predict_proba(X_test)[:, 1]

# Positive uplift = estimated reduction in churn risk
uplift = p_no_offer - p_with_offer

results = pd.DataFrame({
    "Treatment": t_test.to_numpy(),
    "Actual_Synthetic_Churn": y_test.to_numpy(),
    "Predicted_Churn_No_Offer": p_no_offer,
    "Predicted_Churn_With_Offer": p_with_offer,
    "Estimated_Uplift": uplift,
    "True_Uplift": df.loc[
        idx_test, "True_Uplift"
    ].to_numpy(),
    "Potential_Churn_No_Offer": df.loc[
        idx_test, "Potential_Churn_No_Offer"
    ].to_numpy(),
    "Potential_Churn_With_Offer": df.loc[
        idx_test, "Potential_Churn_With_Offer"
    ].to_numpy()
})

print("Mean estimated uplift:", results["Estimated_Uplift"].mean())
print("Mean synthetic true uplift:", results["True_Uplift"].mean())

print(
    "Spearman correlation with synthetic true uplift:",
    spearmanr(
        results["Estimated_Uplift"],
        results["True_Uplift"]
    ).statistic
)

display(
    results.sort_values(
        "Estimated_Uplift", ascending=False
    ).head(10)
)


Mean estimated uplift: 0.08539044607001992
Mean synthetic true uplift: 0.06342640042357371
Spearman correlation with synthetic true uplift: 0.501772989417369


,Treatment,Actual_Synthetic_Churn,Predicted_Churn_No_Offer,Predicted_Churn_With_Offer,Estimated_Uplift,True_Uplift,Potential_Churn_No_Offer,Potential_Churn_With_Offer
676,0,0,0.685932,0.048087,0.637846,0.130161,0,0
207,1,0,0.742482,0.146084,0.596398,0.164693,1,0
294,1,1,0.681943,0.103339,0.578604,0.159826,1,1
661,1,0,0.681943,0.103339,0.578604,0.178489,0,0
944,0,0,0.722686,0.155831,0.566855,0.148940,0,0
365,0,0,0.650248,0.094592,0.555655,0.075634,0,0
55,0,0,0.634792,0.112122,0.522669,0.122022,0,0
526,0,1,0.863459,0.352518,0.510941,0.145306,1,0
1021,1,0,0.682867,0.189420,0.493447,0.137807,0,0
680,0,0,0.719872,0.228972,0.490901,0.125959,0,0


In [48]:
# results is your existing test-set DataFrame.
# Higher True_Uplift means greater synthetic benefit.

results = results.copy()
results["Uplift_Rank"] = results["Estimated_Uplift"].rank(
    ascending=False, method="first"
)

n = max(1, int(np.ceil(0.10 * len(results))))
top_10 = results.nsmallest(n, "Uplift_Rank")

top_10_true = top_10["True_Uplift"].mean()
overall_true = results["True_Uplift"].mean()

print(f"Top 10% mean true uplift: {top_10_true:.4f}")
print(f"Random-targeting benchmark: {overall_true:.4f}")
print(f"Improvement: {top_10_true - overall_true:.4f}")

Top 10% mean true uplift: 0.1253
Random-targeting benchmark: 0.0634
Improvement: 0.0619
